# Massa-mola sem amortecimento (didatico)

Fluxo:
1. Resolver com Euler
2. Mostrar saida numerica
3. Mostrar animacao da mola + grafico x(t)
4. Repetir com RK4


EDO: `x'' + (k/m)*x = 0`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# BLOCO 1: PARAMETROS
m = 1.0
k = 1.0
x0 = 1.0
v0 = 0.0
dt = 0.01
t_final = 20.0
t = np.arange(0.0, t_final + dt, dt)
N = len(t)

# BLOCO 2: EULER
def simular_euler():
    x = np.zeros(N)
    v = np.zeros(N)
    x[0] = x0
    v[0] = v0
    for i in range(N - 1):
        a = -(k / m) * x[i]
        v[i + 1] = v[i] + a * dt
        x[i + 1] = x[i] + v[i] * dt
    return x, v

# BLOCO 3: RK4
def derivadas(x, v):
    return v, -(k / m) * x

def simular_rk4():
    x = np.zeros(N)
    v = np.zeros(N)
    x[0] = x0
    v[0] = v0
    for i in range(N - 1):
        xi, vi = x[i], v[i]
        k1_x, k1_v = derivadas(xi, vi)
        k2_x, k2_v = derivadas(xi + 0.5 * dt * k1_x, vi + 0.5 * dt * k1_v)
        k3_x, k3_v = derivadas(xi + 0.5 * dt * k2_x, vi + 0.5 * dt * k2_v)
        k4_x, k4_v = derivadas(xi + dt * k3_x, vi + dt * k3_v)
        x[i + 1] = xi + (dt / 6.0) * (k1_x + 2 * k2_x + 2 * k3_x + k4_x)
        v[i + 1] = vi + (dt / 6.0) * (k1_v + 2 * k2_v + 2 * k3_v + k4_v)
    return x, v

# BLOCO 4: APOIO

def mostrar_saida(nome, x, v):
    print(f"
Saida numerica - {nome}")
    print(" i   t(s)      x(m)        v(m/s)")
    for i in range(10):
        print(f"{i:2d}  {t[i]:6.3f}   {x[i]:10.6f}   {v[i]:10.6f}")


def desenhar_mola(x_massa, x_parede=-1.2, espiras=12, amplitude=0.08, pontos=220):
    x_final = x_massa - 0.08
    if x_final <= x_parede + 0.05:
        x_final = x_parede + 0.05
    xs = np.linspace(x_parede, x_final, pontos)
    fase = np.linspace(0.0, 2.0 * np.pi * espiras, pontos)
    ys = amplitude * np.sin(fase)
    ys[0] = 0.0
    ys[-1] = 0.0
    return xs, ys


def animar(nome, x):
    fig, (ax_mola, ax_xt) = plt.subplots(2, 1, figsize=(8, 7), gridspec_kw={'height_ratios': [1.3, 1.0]})

    ax_mola.set_title(f'{nome} - Mola oscilando')
    ax_mola.set_xlabel('x (m)')
    ax_mola.set_ylabel('y')
    xmin = min(-1.3, np.min(x) - 0.3)
    xmax = max(1.3, np.max(x) + 0.3)
    ax_mola.set_xlim(xmin, xmax)
    ax_mola.set_ylim(-0.4, 0.4)
    ax_mola.grid(alpha=0.3)
    ax_mola.axvline(-1.2, color='gray', lw=3)

    linha_mola, = ax_mola.plot([], [], color='tab:blue', lw=2)
    massa, = ax_mola.plot([], [], 'o', color='tab:red', ms=14)

    ax_xt.set_title(f'{nome} - Posicao x(t)')
    ax_xt.set_xlabel('tempo (s)')
    ax_xt.set_ylabel('x (m)')
    ax_xt.set_xlim(0.0, t_final)
    margem = 0.1 * max(abs(np.min(x)), abs(np.max(x)), 1e-6)
    ax_xt.set_ylim(np.min(x) - margem, np.max(x) + margem)
    ax_xt.grid(alpha=0.3)
    linha_xt, = ax_xt.plot([], [], color='tab:orange', lw=2)

    fig.subplots_adjust(bottom=0.14, hspace=0.4)
    fig.text(0.02, 0.04, "EDO: x'' + (k/m)*x = 0", fontsize=10)

    def init():
        linha_mola.set_data([], [])
        massa.set_data([], [])
        linha_xt.set_data([], [])
        return linha_mola, massa, linha_xt

    def update(i):
        xs, ys = desenhar_mola(x[i])
        linha_mola.set_data(xs, ys)
        massa.set_data([x[i]], [0.0])
        linha_xt.set_data(t[:i+1], x[:i+1])
        return linha_mola, massa, linha_xt

    ani = FuncAnimation(fig, update, frames=N, init_func=init, interval=12, blit=False)
    plt.close(fig)
    return HTML(ani.to_jshtml())


In [ ]:
# BLOCO 5: EXECUCAO - EULER
x_euler, v_euler = simular_euler()
mostrar_saida('Euler', x_euler, v_euler)
animar('Euler', x_euler)


In [ ]:
# BLOCO 6: EXECUCAO - RK4
x_rk4, v_rk4 = simular_rk4()
mostrar_saida('Runge-Kutta 4', x_rk4, v_rk4)
animar('Runge-Kutta 4', x_rk4)
